# Module 09 — Graph Algorithms Shortest Paths and MST

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_dijkstra import dijkstra
from p02_bellman_ford import bellman_ford
from p03_network_delay import network_delay

print("module 09: Graph Algorithms Shortest Paths and MST")
print("problems available:", 8)
for name in ['p01_dijkstra', 'p02_bellman_ford', 'p03_network_delay', 'p04_cheapest_flights_k_stops', 'p05_kruskal_mst', 'p06_prim_mst', 'p07_redundant_connection', 'p08_min_cost_connect_points']:
    print(f"  {name}")

## 1. Baseline — `p01_dijkstra`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
assert dijkstra(3, [(0, 1, 4), (0, 2, 1), (2, 1, 2)], 0) == [0, 3, 1]
# Unreachable nodes stay at infinity.
assert dijkstra(3, [(0, 1, 1)], 0) == [0, 1, float('inf')]
assert dijkstra(1, [], 0) == [0]
# A zero-weight edge is legal.
assert dijkstra(2, [(0, 1, 0)], 0) == [0, 0]
# Parallel edges: the cheaper one must win.
assert dijkstra(2, [(0, 1, 5), (0, 1, 2)], 0) == [0, 2]
# A self-loop changes nothing.
assert dijkstra(2, [(0, 0, 3), (0, 1, 1)], 0) == [0, 1]
# The direct edge is not always the shortest route.
assert dijkstra(4, [(0, 1, 10), (0, 2, 1), (2, 3, 1), (3, 1, 1)], 0) == [0, 3, 1, 2]
# Starting elsewhere.
assert dijkstra(3, [(0, 1, 1), (1, 2, 1)], 1) == [float('inf'), 0, 1]
# Scale: a 50k chain.
chain = [(i, i + 1, 1) for i in range(50_000)]
d = dijkstra(50_001, chain, 0)
assert d[-1] == 50_000 and d[0] == 0

print("all assertions held")

## 2. Predict before you run

A graph has edges 0→1 costing 4, 0→2 costing 1, and 2→1 costing -2. Predict the shortest distance from 0 to 1. Then predict what Dijkstra returns. Those two numbers differ, and nothing raises.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
# A negative edge that Dijkstra would get wrong.
assert bellman_ford(3, [(0, 1, 4), (0, 2, 1), (2, 1, -2)], 0) == [0, -1, 1]
# A reachable negative cycle.
assert bellman_ford(2, [(0, 1, 1), (1, 0, -3)], 0) is None
assert bellman_ford(1, [], 0) == [0]
# All non-negative: must agree with Dijkstra.
edges = [(0, 1, 4), (0, 2, 1), (2, 1, 2), (1, 3, 1), (2, 3, 5)]
assert bellman_ford(4, edges, 0) == dijkstra(4, edges, 0)
# Unreachable nodes stay at infinity.
assert bellman_ford(3, [(0, 1, 2)], 0) == [0, 2, float('inf')]
# An UNREACHABLE negative cycle is not an error - the source's own, # distances are still well defined.
res = bellman_ford(4, [(0, 1, 1), (2, 3, -1), (3, 2, -1)], 0)
assert res is not None
assert res[0] == 0 and res[1] == 1
# A negative self-loop is a negative cycle.
assert bellman_ford(2, [(0, 1, 1), (1, 1, -1)], 0) is None
# A single negative edge with no cycle is fine.
assert bellman_ford(2, [(0, 1, -5)], 0) == [0, -5]

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert network_delay(4, [(2, 1, 1), (2, 3, 1), (3, 4, 1)], 2) == 2
assert network_delay(2, [(1, 2, 1)], 1) == 1
# Node 1 is unreachable from node 2.
assert network_delay(2, [(1, 2, 1)], 2) == -1
# A single node needs no time.
assert network_delay(1, [], 1) == 0
# It is the maximum, not the sum.
assert network_delay(3, [(1, 2, 1), (1, 3, 5)], 1) == 5
# A detour can beat the direct edge.
assert network_delay(3, [(1, 2, 10), (1, 3, 1), (3, 2, 1)], 1) == 2
# Zero-weight edges.
assert network_delay(2, [(1, 2, 0)], 1) == 0
# Disconnected node.
assert network_delay(3, [(1, 2, 1)], 1) == -1

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. The edge weights choose the algorithm: BFS, Dijkstra, or Bellman-Ford.
2. Dijkstra on a negative edge does not error. It returns a larger number, confidently.
3. Bellman-Ford's extra round is what makes the answer meaningful rather than arbitrary.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem